In [ ]:
# 安装依赖（如果未安装，取消注释运行）
# !pip install vnpy vnpy_tqsdk vnpy_ctp vnpy_ctastrategy pandas

import pandas as pd
from datetime import datetime, timedelta
from vnpy.event import EventEngine
from vnpy.trader.engine import MainEngine
from vnpy.trader.object import SubscribeRequest, HistoryRequest
from vnpy.trader.constant import Exchange, Interval
from vnpy_tqsdk import Datafeed  # 天勤：历史数据
from vnpy_ctp import CtpGateway      # CTP：模拟盘
from vnpy_ctastrategy import CtaStrategyApp
from vnpy_ctastrategy.backtesting import BacktestingEngine
from vnpy.trader.datafeed import get_datafeed
import time
import logging  # 用于日志记录

In [ ]:
# 天勤配置：替换为您的账号
TQ_CONFIG = {
    "username": "your_tq_username",
    "password": "your_tq_password"
}

# SimNow配置：替换为您的账号（实盘模拟环境示例）
SIMNOW_CONFIG = {
    "td_address": "tcp://180.168.146.187:10130",  # 交易服务器
    "md_address": "tcp://180.168.146.187:10131",  # 行情服务器
    "brokerid": "9999",                           # 经纪商代码
    "userid": "your_simnow_userid",
    "password": "your_simnow_password"
}

# 回测参数（可调整）
BACKTEST_PARAMS = {
    "vt_symbol": "ag8888.SHFE",
    "interval": Interval.TICK,
    "start": datetime.now() - timedelta(days=14),
    "end": datetime.now(),
    "rate": 0.0001,  # 手续费率
    "slippage": 0.2, # 滑点
    "size": 15,      # 沪银合约乘数（标准15手）
    "pricetick": 1,  # 最小变动价位
    "capital": 1_000_000  # 初始资金
}

In [ ]:
# 导入策略类（假设 strategies 文件夹在同一目录）
from strategies.simple_ag_strategy_real_time import SimpleAgStrategy

In [ ]:
def download_tq_history(main_engine: MainEngine):
    """使用天勤下载当前时间前14天沪银主连历史Tick"""
    start = datetime.now() - timedelta(days=14)
    end = datetime.now()

    req = HistoryRequest(
        symbol="ag8888",
        exchange=Exchange.SHFE,
        start=start,
        end=end,
        interval=Interval.TICK  # Tick级别（需专业版权限）
    )

    datafeed = get_datafeed()  # 天勤数据服务
    ticks = datafeed.query_tick_history(req)

    if ticks:
        df = pd.DataFrame([tick.__dict__ for tick in ticks])
        df['datetime'] = pd.to_datetime(df['datetime'])
        df.set_index('datetime', inplace=True)
        hdf_path = "ag8888_tq_history_14d.h5"
        df.to_hdf(hdf_path, key='tick_data')
        print(f"天勤历史数据下载完成：{len(ticks)} 条 Tick，保存至 {hdf_path}")
        return hdf_path, ticks  # 返回路径和原始 ticks 用于回测
    else:
        print("无历史数据")
        return None, None

In [ ]:
def run_backtest(main_engine: MainEngine, hdf_path: str, ticks):
    """使用历史数据运行回测，计算统计，并记录交易日志"""
    if not hdf_path:
        print("无历史数据，无法回测")
        return

    # 初始化回测引擎
    backtesting_engine = BacktestingEngine()

    # 设置参数
    backtesting_engine.set_parameters(
        vt_symbol=BACKTEST_PARAMS["vt_symbol"],
        interval=BACKTEST_PARAMS["interval"],
        start=BACKTEST_PARAMS["start"],
        end=BACKTEST_PARAMS["end"],
        rate=BACKTEST_PARAMS["rate"],
        slippage=BACKTEST_PARAMS["slippage"],
        size=BACKTEST_PARAMS["size"],
        pricetick=BACKTEST_PARAMS["pricetick"],
        capital=BACKTEST_PARAMS["capital"]
    )

    # 添加策略
    backtesting_engine.add_strategy(SimpleAgStrategy, {})

    # 加载历史数据（从 ticks 转换到 BarData，简化处理；实际可优化为 Tick 回测）
    from vnpy.trader.object import BarData
    bars = []
    for tick in ticks:
        bar = BarData(
            symbol=tick.symbol,
            exchange=tick.exchange,
            datetime=tick.datetime,
            interval=Interval.TICK,
            volume=tick.volume,
            open_interest=tick.open_interest,
            open_price=tick.last_price,  # 简化：用 last_price 填充
            high_price=tick.last_price,
            low_price=tick.last_price,
            close_price=tick.last_price
        )
        bars.append(bar)
    backtesting_engine.load_bar_data(bars)

    # 运行回测
    backtesting_engine.run_backtesting()

    # 计算统计
    stats = backtesting_engine.calculate_statistics()
    print("回测统计：")
    print(f"总收益率: {stats['total_return']}%")
    print(f"夏普比率: {stats['sharpe_ratio']}")
    print(f"最大回撤: {stats['max_drawdown']}%")
    print(f"交易次数: {stats['total_trade_count']}")
    # 更多统计：stats['calmar_ratio'], stats['sortino_ratio'] 等

    # 记录交易日志
    trades = backtesting_engine.get_all_trades()
    with open("backtest_trades.log", "w") as log_file:
        log_file.write("回测交易日志:\n")
        for trade in trades:
            log_str = f"{trade.datetime} - {trade.direction} {trade.volume} @ {trade.price} (Offset: {trade.offset})\n"
            log_file.write(log_str)
            print(log_str)  # 同时打印到控制台
    print(f"交易日志保存至 backtest_trades.log")

In [ ]:
def main():
    event_engine = EventEngine()
    main_engine = MainEngine(event_engine)

    # 第一步：添加天勤Gateway，仅用于历史数据下载
    main_engine.add_gateway(Datafeed)
    main_engine.connect(TQ_CONFIG, "tqsdk")
    hdf_path, ticks = download_tq_history(main_engine)

    # 第二步：运行回测（使用历史数据）
    run_backtest(main_engine, hdf_path, ticks)

    # 第三步：添加CTP Gateway，连接SimNow模拟盘
    main_engine.add_gateway(CtpGateway)
    main_engine.add_app(CtaStrategyApp)
    main_engine.connect(SIMNOW_CONFIG, "ctp")

    # 添加策略（沪银主连）
    strategy = main_engine.add_engine(SimpleAgStrategy)

    # 订阅模拟盘合约
    req = SubscribeRequest(
        symbol="ag8888",
        exchange=Exchange.SHFE
    )
    main_engine.subscribe(req, "ctp")

    # 启动策略
    strategy.inited = True
    strategy.trading = True

    print("SimNow模拟盘订阅启动，按 Ctrl+C 停止...")
    try:
        while True:
            time.sleep(1)  # 监听Tick
    except KeyboardInterrupt:
        strategy.stop()
        main_engine.close()

In [ ]:
if __name__ == "__main__":
    main()